# Databases and data systems: local, remote, and larger than memory

Design, query, validate, and operate scientific data across SQLite, DuckDB, pandas, PyArrow,
Polars, SQLAlchemy, AWS-style object storage, and a PostgreSQL client/server boundary.

**Lecture 6 · Notebook 00 · CMOR 438 / INDE 577**

## Orientation: data persistence is a system boundary

**Live core:** relational thinking, SQLite, constraints, parameterized SQL, transactions, joins,
aggregation, pandas integration, analytical files, working-set estimates, and data pushdown.

**Practice:** diagnose unsafe queries, design schemas, process bounded batches, choose a scalable
architecture, and build a reproducible training-data extract.

**Extension:** DuckDB, PyArrow, Polars lazy plans, AWS S3/Athena workflows, query plans, pools,
isolation, migrations, governance, observability, and operational design.

Every executable database and data lake is disposable and local. AWS and PostgreSQL examples use
offline stubs or construct client configuration without making a network call, so the notebook is
deterministic in CI and requires no cloud account or credentials.

## How to use this notebook

**Estimated time:** 225 minutes core, plus 180 minutes of practice and extension. Treat this as a reference notebook: the instructor will choose a three-hour live route.

**Prerequisites:** Python contexts and exceptions, native files, pandas, testing, packages, and the
Rice DSM environment. Run `uv sync`, select the **Rice DSM** kernel in VS Code, then restart and run all.

Read SQL as a program with inputs, outputs, assumptions, side effects, and failure modes. Before each
query, predict its rows, columns, types, ordering, missing-value behavior, transaction effect, and
cost. Database syntax varies: confirm behavior against the actual engine used in production.

## Learning objectives

By the end, you should be able to:

- distinguish a database, database-management system, driver, connection, cursor, transaction, and
  SQLAlchemy engine;
- compare local SQLite, in-process DuckDB, and remote client/server databases;
- design tables with primary keys, foreign keys, uniqueness, nullability, and check constraints;
- create and populate a SQLite database using explicit, parameterized transactions;
- query safely with filtering, joins, grouping, common-table expressions, and window functions;
- explain SQL `NULL` and how its semantics differ from ordinary Python values;
- move deliberately between SQL results and pandas without hiding types or row ordering;
- inspect indexes and query plans without assuming that an index always improves performance;
- use SQLAlchemy connections, transactions, metadata, and dialect-aware statements;
- estimate a working set and choose projection, filtering, batching, lazy execution, or distributed compute before materializing data;
- scan partitioned Parquet with PyArrow, Polars, and DuckDB while pushing columns and predicates toward storage;
- distinguish S3 object storage, Athena, RDS, Redshift, and a metadata catalog in a typical AWS analytical workflow;
- configure AWS and PostgreSQL clients without embedding or printing credentials;
- reason about pooling, timeouts, TLS, retries, isolation, migrations, and least privilege; and
- make a data extract reproducible through query text, parameters, schema version, snapshot identity,
  ordering, and validation.

## Why this matters

Notebooks often begin with a CSV but production data rarely remains one file. Databases coordinate
durable state, concurrent users, constraints, access control, transactions, and efficient queries.
They also create new risks: SQL injection, leaked credentials, partial writes, inconsistent reads,
schema drift, accidental full-table scans, silent row multiplication, and irreproducible “latest data.”

For machine learning, the database query is part of the experiment. A perfectly versioned model is
not reproducible if its training rows came from an unrecorded mutable query result.

## Worked example: a scientific measurement repository

We model two heat experiments, three sensors, and repeated temperature measurements:

```text
experiments 1 ─────< measurements >───── 1 sensors
 experiment_id          experiment_id          sensor_id
 material               sensor_id              distance_cm
 started_at              time_s                 sensor_model
 operator_code           temperature_c
                         quality_flag
```

The schema separates experimental context, sensor metadata, and observations. Foreign keys make the
relationships executable. The data are synthetic teaching records, not empirical material evidence.

## Professional practice

| Data scientist asks | Software engineer asks |
| --- | --- |
| What does one row represent? | Which key makes that identity enforceable? |
| Which snapshot produced this analysis? | Where are query, parameters, and schema version recorded? |
| Could the join duplicate or lose observations? | Which cardinality and row-count checks protect it? |
| Does `NULL` mean missing, failed, or not applicable? | Which column, constraint, and application policy owns it? |
| Could future information enter training rows? | Which query boundary and test prevent leakage? |
| Is the aggregate scientifically meaningful? | Which transaction, isolation level, and ordering make it repeatable? |
| Who should access sensitive fields? | Which role, grant, audit, and retention policy applies? |

The database is not merely storage. Its schema and query behavior participate in scientific validity.

## 1. Vocabulary and deployment shapes

- A **database** is an organized body of data and metadata.
- A **DBMS** is software that stores, queries, protects, and coordinates that data.
- A **driver** implements a language-specific API (application programming interface) for speaking
  the DBMS protocol; our Python code calls the driver instead of constructing network bytes itself.
- A **connection** is one active conversation with a database.
- A **cursor/result** executes or traverses statement results.
- A **transaction** groups operations into an atomic success or rollback boundary.
- A **schema** names tables, columns, types, constraints, views, and relationships.
- A SQLAlchemy **Engine** combines a database dialect, driver, and usually a connection pool.

“SQL database” does not imply identical SQL, types, concurrency, or operational behavior.

### Local and remote are deployment properties, not quality rankings

| Shape | Examples | Strengths | Important limits |
| --- | --- | --- | --- |
| embedded file/serverless | SQLite | standard library, portable file, transactions | limited write concurrency, engine-specific typing |
| in-process analytical | DuckDB | vectorized analytical SQL, files/DataFrames | not a general multi-user transaction server |
| remote client/server | PostgreSQL, MySQL | concurrency, roles, network clients, operations | latency, credentials, service ownership, cost |
| managed warehouse/lakehouse | BigQuery, Snowflake, Databricks | scalable analytics and governance | vendor semantics, spend, data movement |

Use the smallest system that satisfies concurrency, durability, scale, governance, and availability.
Do not use SQLite as a drop-in behavioral substitute for every production database.

## 2. Confirm the database environment

Python includes `sqlite3`. DuckDB, SQLAlchemy, Psycopg, Boto3, PyArrow, and Polars are declared
runtime dependencies and locked with the course environment. Psycopg is a PostgreSQL driver and
Boto3 is the AWS SDK for Python; installing either one does not create a server or cloud account.

In [ ]:
import sqlite3
from collections.abc import Iterator
from math import isclose
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import urlsplit

import boto3
import duckdb
import pandas as pd
import polars as pl
import psycopg
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as pads
import sqlalchemy
from botocore import UNSIGNED
from botocore.config import Config
from botocore.stub import Stubber
from sqlalchemy import URL, MetaData, Table, create_engine, inspect, select, text
from sqlalchemy.engine import Engine

print("SQLite runtime:", sqlite3.sqlite_version)
print("DuckDB:       ", duckdb.__version__)
print("SQLAlchemy:   ", sqlalchemy.__version__)
print("Psycopg:      ", psycopg.__version__)
print("Boto3:        ", boto3.__version__)
print("PyArrow:      ", pa.__version__)
print("Polars:       ", pl.__version__)

assert sqlite3.sqlite_version_info >= (3, 35, 0)
assert int(duckdb.__version__.split(".")[0]) >= 1
assert int(sqlalchemy.__version__.split(".")[0]) >= 2
assert int(psycopg.__version__.split(".")[0]) >= 3
assert int(boto3.__version__.split(".")[0]) >= 1
assert int(pa.__version__.split(".")[0]) >= 20
assert int(pl.__version__.split(".")[0]) >= 1

## 3. Create a disposable SQLite database

`":memory:"` creates a database tied to one connection. A file path persists across connections. We
use a temporary directory so students experience real file persistence without leaving artifacts or
depending on an operating-system-specific path.

In [ ]:
database_workspace = TemporaryDirectory()
database_path = Path(database_workspace.name) / "scientific-measurements.sqlite3"

sqlite_connection = sqlite3.connect(database_path)
sqlite_connection.row_factory = sqlite3.Row
sqlite_connection.execute("PRAGMA foreign_keys = ON")

assert database_path.is_file()
assert sqlite_connection.execute("PRAGMA foreign_keys").fetchone()[0] == 1
print("Disposable database:", database_path.name)

SQLite foreign-key enforcement must be enabled for each connection. A schema declaration that the
engine does not enforce is false confidence. Connection setup belongs in one tested factory or engine
hook rather than being remembered throughout notebooks.

## 4. Schema design makes important invariants executable

Keys should describe domain identity rather than accidental row order. Constraints reject invalid
state at the shared boundary:

- `PRIMARY KEY`: unique, non-null row identity;
- `FOREIGN KEY`: referenced identity must exist;
- `UNIQUE`: candidate key may not repeat;
- `NOT NULL`: missing is not permitted;
- `CHECK`: engine-checkable domain predicate; and
- `DEFAULT`: value supplied when omitted, not a repair for invalid input.

Constraints complement application validation. They cannot encode every scientific rule.

In [ ]:
schema_sql = """
CREATE TABLE experiments (
    experiment_id INTEGER PRIMARY KEY,
    material TEXT NOT NULL CHECK (material IN ('copper', 'aluminum')),
    started_at TEXT NOT NULL,
    operator_code TEXT NOT NULL
);

CREATE TABLE sensors (
    sensor_id TEXT PRIMARY KEY,
    distance_cm REAL NOT NULL CHECK (distance_cm > 0),
    sensor_model TEXT NOT NULL
);

CREATE TABLE measurements (
    measurement_id INTEGER PRIMARY KEY,
    experiment_id INTEGER NOT NULL,
    sensor_id TEXT NOT NULL,
    time_s INTEGER NOT NULL CHECK (time_s >= 0),
    temperature_c REAL,
    quality_flag TEXT NOT NULL
        CHECK (quality_flag IN ('ok', 'sensor_dropout', 'review')),
    UNIQUE (experiment_id, sensor_id, time_s),
    FOREIGN KEY (experiment_id) REFERENCES experiments(experiment_id),
    FOREIGN KEY (sensor_id) REFERENCES sensors(sensor_id),
    CHECK (
        (temperature_c IS NOT NULL AND quality_flag IN ('ok', 'review'))
        OR (temperature_c IS NULL AND quality_flag = 'sensor_dropout')
    )
);
"""

with sqlite_connection:
    sqlite_connection.executescript(schema_sql)

table_names = {
    row[0]
    for row in sqlite_connection.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table'"
    )
}
assert {"experiments", "sensors", "measurements"} <= table_names

`executescript` is appropriate here because the DDL is trusted static source code. Never place
untrusted input into a SQL script. Production schema changes belong in ordered, reviewed migrations,
not ad hoc notebook cells.

### Normalization reduces contradictory copies

Sensor distance appears once in `sensors`, not on every measurement. Updating one sensor record does
not leave competing distances across rows. Normalization improves integrity; analytical systems may
also use deliberate denormalization for performance. Duplication needs an ownership and refresh rule.

## 5. Insert values through parameters, never string formatting

SQL text and values travel separately. For Python's SQLite driver, `?` placeholders represent values.
Placeholders protect quoting and type adaptation and prevent a value from becoming executable SQL.
They cannot stand in for a table or column identifier.

In [ ]:
experiment_rows = [
    (1, "copper", "2026-08-29T09:00:00-05:00", "op-17"),
    (2, "aluminum", "2026-08-29T13:00:00-05:00", "op-22"),
]
sensor_rows = [
    ("S1", 1.0, "T-100"),
    ("S2", 2.5, "T-100"),
    ("S3", 5.0, "T-200"),
]

with sqlite_connection:
    sqlite_connection.executemany(
        """
        INSERT INTO experiments (
            experiment_id, material, started_at, operator_code
        ) VALUES (?, ?, ?, ?)
        """,
        experiment_rows,
    )
    sqlite_connection.executemany(
        """
        INSERT INTO sensors (sensor_id, distance_cm, sensor_model)
        VALUES (?, ?, ?)
        """,
        sensor_rows,
    )

assert sqlite_connection.execute("SELECT COUNT(*) FROM experiments").fetchone()[0] == 2
assert sqlite_connection.execute("SELECT COUNT(*) FROM sensors").fetchone()[0] == 3

`executemany` sends many parameter sets through one statement shape. It is clearer and usually more
efficient than executing one insert in a Python loop. For very large loads, use the database's bulk
loading mechanism and validate staging tables before promotion.

In [ ]:
measurement_rows: list[tuple[int, str, int, float | None, str]] = []
base_temperatures_c = {
    1: [74.2, 65.1, 56.0, 48.0],
    2: [62.0, 54.5, 47.8, 41.9],
}
sensor_adjustments_c = {"S1": 1.5, "S2": 0.0, "S3": -2.2}

for experiment_id, temperatures_c in base_temperatures_c.items():
    for time_s, base_temperature_c in zip(
        [0, 30, 60, 90],
        temperatures_c,
        strict=True,
    ):
        for sensor_id, adjustment_c in sensor_adjustments_c.items():
            temperature_c: float | None = base_temperature_c + adjustment_c
            quality_flag = "ok"
            if experiment_id == 2 and time_s == 60 and sensor_id == "S3":
                temperature_c = None
                quality_flag = "sensor_dropout"
            measurement_rows.append(
                (
                    experiment_id,
                    sensor_id,
                    time_s,
                    temperature_c,
                    quality_flag,
                )
            )

with sqlite_connection:
    sqlite_connection.executemany(
        """
        INSERT INTO measurements (
            experiment_id, sensor_id, time_s, temperature_c, quality_flag
        ) VALUES (?, ?, ?, ?, ?)
        """,
        measurement_rows,
    )

assert len(measurement_rows) == 24
stored_measurement_count = sqlite_connection.execute(
    "SELECT COUNT(*) FROM measurements"
).fetchone()[0]
assert stored_measurement_count == 24

### Unsafe construction changes the program

This string is shown but never executed:

```python
unsafe_sql = f"SELECT * FROM experiments WHERE material = '{user_value}'"
```

If `user_value` contains quotes and SQL syntax, it changes the statement. Escaping by hand is not a
reliable repair. Bind values with the driver's parameter API. For dynamic identifiers, choose from an
application-owned allowlist or use the driver's identifier-composition facility.

In [ ]:
requested_material = "copper' OR 1=1 --"
unsafe_sql = (
    "SELECT * FROM experiments WHERE material = "
    f"'{requested_material}'"
)
safe_sql = "SELECT * FROM experiments WHERE material = ?"

print("Unsafe statement would become:", unsafe_sql)
safe_rows = sqlite_connection.execute(safe_sql, (requested_material,)).fetchall()

assert "OR 1=1" in unsafe_sql
assert safe_rows == []

Do not log the unsafe statement when it may contain secrets or personal data. Parameterization is a
security boundary and a correctness mechanism, not merely cleaner syntax.

## 6. Query results have shape, types, and ordering

SQL tables have no guaranteed presentation order. Use `ORDER BY` whenever order is part of the output
contract. Select named columns instead of `SELECT *` in stable interfaces: adding or reordering a
database column should not silently reorder model features.

In [ ]:
measurement_query = """
SELECT
    m.measurement_id,
    e.material,
    m.time_s,
    m.sensor_id,
    s.distance_cm,
    m.temperature_c,
    m.quality_flag
FROM measurements AS m
JOIN experiments AS e USING (experiment_id)
JOIN sensors AS s USING (sensor_id)
WHERE e.material = :material
  AND m.time_s >= :minimum_time_s
ORDER BY m.time_s, m.sensor_id
"""

query_parameters = {"material": "copper", "minimum_time_s": 60}
selected_rows = sqlite_connection.execute(
    measurement_query,
    query_parameters,
).fetchall()

assert len(selected_rows) == 6
assert list(selected_rows[0].keys()) == [
    "measurement_id",
    "material",
    "time_s",
    "sensor_id",
    "distance_cm",
    "temperature_c",
    "quality_flag",
]
assert [(row["time_s"], row["sensor_id"]) for row in selected_rows] == sorted(
    (row["time_s"], row["sensor_id"]) for row in selected_rows
)

Named parameters can be easier to review when a statement has several inputs. Placeholder syntax is
driver-specific: SQLite supports `?` and named styles; Psycopg commonly uses `%s`; SQLAlchemy `text()`
uses `:name` and translates through its dialect.

## 7. SQL `NULL` uses three-valued logic

`NULL` represents an absent/unknown SQL value, not Python `None`, `NaN`, zero, or an empty string,
although drivers may convert SQL `NULL` to Python `None`. Comparisons such as `temperature_c = NULL`
evaluate to unknown, not true. Use `IS NULL` and `IS NOT NULL`.

In [ ]:
wrong_null_count = sqlite_connection.execute(
    "SELECT COUNT(*) FROM measurements WHERE temperature_c = NULL"
).fetchone()[0]
correct_null_count = sqlite_connection.execute(
    "SELECT COUNT(*) FROM measurements WHERE temperature_c IS NULL"
).fetchone()[0]

assert wrong_null_count == 0
assert correct_null_count == 1

Aggregates have distinct missing rules. `COUNT(*)` counts rows, `COUNT(column)` counts non-null values,
and `AVG(column)` normally ignores null values. Always report denominators when missingness matters.

In [ ]:
aggregate_rows = sqlite_connection.execute(
    """
    SELECT
        e.material,
        COUNT(*) AS total_rows,
        COUNT(m.temperature_c) AS valid_temperatures,
        COUNT(*) - COUNT(m.temperature_c) AS missing_temperatures,
        AVG(m.temperature_c) AS mean_temperature_c
    FROM measurements AS m
    JOIN experiments AS e USING (experiment_id)
    GROUP BY e.material
    ORDER BY e.material
    """
).fetchall()

aggregate_by_material = {row["material"]: dict(row) for row in aggregate_rows}
assert aggregate_by_material["copper"]["valid_temperatures"] == 12
assert aggregate_by_material["aluminum"]["valid_temperatures"] == 11
assert aggregate_by_material["aluminum"]["missing_temperatures"] == 1

## 8. Joins encode cardinality assumptions

An inner join keeps matching rows. Left, right, and full joins make different retention promises.
Unexpected duplicate keys can multiply output rows; unmatched keys can drop observations. Before and
after a join, check key uniqueness, expected cardinality, row counts, and unmatched identifiers.

In [ ]:
join_count = sqlite_connection.execute(
    """
    SELECT COUNT(*)
    FROM measurements AS m
    JOIN experiments AS e USING (experiment_id)
    JOIN sensors AS s USING (sensor_id)
    """
).fetchone()[0]

orphan_count = sqlite_connection.execute(
    """
    SELECT COUNT(*)
    FROM measurements AS m
    LEFT JOIN sensors AS s USING (sensor_id)
    WHERE s.sensor_id IS NULL
    """
).fetchone()[0]

assert join_count == 24
assert orphan_count == 0

Foreign keys prevent new orphan records through this connection, but historical imports, disabled
constraint enforcement, or cross-system joins can still violate expectations. Validate the result,
not only the schema diagram.

## 9. Transactions protect multi-step state changes

Atomicity means a transaction's changes commit together or roll back together. The Python SQLite
connection context manager commits when its block exits normally and rolls back when an exception
escapes. It does **not** close the connection.

In [ ]:
before_count = sqlite_connection.execute(
    "SELECT COUNT(*) FROM experiments"
).fetchone()[0]

try:
    with sqlite_connection:
        sqlite_connection.execute(
            """
            INSERT INTO experiments (
                experiment_id, material, started_at, operator_code
            ) VALUES (?, ?, ?, ?)
            """,
            (99, "copper", "2026-08-30T09:00:00-05:00", "op-test"),
        )
        raise RuntimeError("simulate a later application failure")
except RuntimeError as error:
    print("Transaction rolled back:", error)

after_count = sqlite_connection.execute(
    "SELECT COUNT(*) FROM experiments"
).fetchone()[0]

assert after_count == before_count
assert sqlite_connection.execute(
    "SELECT COUNT(*) FROM experiments WHERE experiment_id = ?",
    (99,),
).fetchone()[0] == 0

Catch the exception outside the transaction boundary. Swallowing it inside can cause the context to
commit. External side effects—email, object storage, another service—do not automatically roll back
with the database; distributed consistency needs an explicit design.

### Constraints make invalid transactions fail

Application validation gives early domain-specific messages. Database constraints remain the final
shared defense against every writer.

In [ ]:
try:
    with sqlite_connection:
        sqlite_connection.execute(
            """
            INSERT INTO measurements (
                experiment_id, sensor_id, time_s, temperature_c, quality_flag
            ) VALUES (?, ?, ?, ?, ?)
            """,
            (1, "S1", -1, 55.0, "ok"),
        )
except sqlite3.IntegrityError as error:
    print(type(error).__name__, error)
else:
    raise AssertionError("negative time should violate the schema")

Do not depend on exact database error text across engines and versions. Translate errors only at a
layer that can add useful context while preserving the original cause.

## 10. Common-table expressions and window functions

A **CTE** names an intermediate query. A **window function** computes across related rows without
collapsing them into one group row. Here `LAG` exposes change from the previous time for each
experiment–sensor sequence.

In [ ]:
change_query = """
WITH ordered_measurements AS (
    SELECT
        experiment_id,
        sensor_id,
        time_s,
        temperature_c,
        LAG(temperature_c) OVER (
            PARTITION BY experiment_id, sensor_id
            ORDER BY time_s
        ) AS previous_temperature_c
    FROM measurements
)
SELECT
    experiment_id,
    sensor_id,
    time_s,
    temperature_c,
    temperature_c - previous_temperature_c AS change_c
FROM ordered_measurements
ORDER BY experiment_id, sensor_id, time_s
"""

change_rows = sqlite_connection.execute(change_query).fetchall()

assert len(change_rows) == 24
assert sum(row["change_c"] is None for row in change_rows) == 8

There are six partition starts. The dropout makes its own change null, and the following observation
also has no numerical predecessor, hence eight null changes. Window
frames, ties, and ordering require careful review. If time can repeat, add a deterministic tie-breaker
or decide how simultaneous records should behave.

## 11. Indexes trade write/storage cost for access paths

An index is a data structure the optimizer may use to locate or order rows. It consumes space and
makes writes more expensive. Column order matters in composite indexes. An optimizer may correctly
prefer a full scan for small tables or low-selectivity predicates.

In [ ]:
with sqlite_connection:
    sqlite_connection.execute(
        """
        CREATE INDEX idx_measurements_sensor_time
        ON measurements (sensor_id, time_s)
        """
    )

query_plan = sqlite_connection.execute(
    """
    EXPLAIN QUERY PLAN
    SELECT temperature_c
    FROM measurements
    WHERE sensor_id = ? AND time_s >= ?
    ORDER BY time_s
    """,
    ("S2", 30),
).fetchall()
plan_text = " ".join(str(value) for row in query_plan for value in row)

print(plan_text)
assert "idx_measurements_sensor_time" in plan_text

Use `EXPLAIN`/`EXPLAIN ANALYZE` on representative production-like data. Timings from 24 local rows do
not predict remote performance. Query plans can change with engine version, statistics, parameters,
data distribution, indexes, and configuration.

## 12. pandas reads query results—not an abstract database truth

`pd.read_sql_query` executes supplied SQL through a supported connection and materializes a DataFrame.
That transfer consumes client memory and loses some database-specific type/constraint information.
Push selective filters and aggregations into the database when appropriate, then validate the result.

In [ ]:
measurement_frame = pd.read_sql_query(
    measurement_query,
    sqlite_connection,
    params=query_parameters,
)

assert measurement_frame.shape == (6, 7)
assert measurement_frame.columns.tolist() == [
    "measurement_id",
    "material",
    "time_s",
    "sensor_id",
    "distance_cm",
    "temperature_c",
    "quality_flag",
]
assert measurement_frame["temperature_c"].notna().all()
measurement_frame

SQL results are not guaranteed ordered merely because a previous run looked stable. This query's
`ORDER BY m.time_s, m.sensor_id` makes row order part of the extract contract.

### Chunking bounds client memory but changes the processing interface

`chunksize` returns an iterator of DataFrames. Cross-chunk operations require state or a database-side
aggregation. A chunk is not a statistically random sample.

In [ ]:
chunk_iterator = pd.read_sql_query(
    """
    SELECT measurement_id, experiment_id, sensor_id, time_s, temperature_c
    FROM measurements
    ORDER BY measurement_id
    """,
    sqlite_connection,
    chunksize=7,
)
measurement_chunks = list(chunk_iterator)

assert [len(chunk) for chunk in measurement_chunks] == [7, 7, 7, 3]
assert sum(len(chunk) for chunk in measurement_chunks) == 24

### `DataFrame.to_sql` is useful, but inferred schema is not governance

Writing a DataFrame can create a convenient staging table. For production tables, define types,
constraints, indexes, ownership, and migrations explicitly. `if_exists="replace"` is destructive and
should not target shared production data casually.

In [ ]:
analysis_summary = pd.DataFrame(
    [dict(row) for row in aggregate_rows]
)
analysis_summary.to_sql(
    "material_summary_staging",
    sqlite_connection,
    if_exists="fail",
    index=False,
)

staging_count = sqlite_connection.execute(
    "SELECT COUNT(*) FROM material_summary_staging"
).fetchone()[0]
assert staging_count == 2

## 13. SQLAlchemy separates dialect, connection, and statement concerns

SQLAlchemy does not make every database identical. Its Engine provides a consistent connection and
transaction boundary, delegates engine-specific behavior to a dialect/driver, and usually manages a
pool for client/server databases.

In [ ]:
sqlite_url = URL.create(
    "sqlite+pysqlite",
    database=str(database_path),
)
sqlalchemy_engine = create_engine(sqlite_url)

assert isinstance(sqlalchemy_engine, Engine)
assert sqlalchemy_engine.dialect.name == "sqlite"
assert sqlalchemy_engine.url.database == str(database_path)

Creating an Engine is lazy: it normally does not open the first DBAPI connection until work needs one.
Create one long-lived Engine per database/application process rather than one per query. Check the
database driver's thread/process rules before sharing it.

In [ ]:
with sqlalchemy_engine.connect() as connection:
    result = connection.execute(
        text(
            """
            SELECT material, started_at
            FROM experiments
            WHERE material = :material
            """
        ),
        {"material": "copper"},
    )
    experiment_mapping = result.mappings().one()

assert experiment_mapping["material"] == "copper"
assert experiment_mapping["started_at"].startswith("2026-08-29")

`Connection.execute(text(...), parameters)` keeps values separate from SQL. Result mappings support
column-name access. A connection context closes/returns the connection; a transaction context also
decides commit or rollback.

In [ ]:
with sqlalchemy_engine.begin() as connection:
    connection.execute(
        text(
            """
            CREATE TABLE IF NOT EXISTS analysis_runs (
                run_id TEXT PRIMARY KEY,
                query_name TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
            """
        )
    )
    connection.execute(
        text(
            """
            INSERT INTO analysis_runs (run_id, query_name, created_at)
            VALUES (:run_id, :query_name, :created_at)
            """
        ),
        {
            "run_id": "run-001",
            "query_name": "copper-late-measurements-v1",
            "created_at": "2026-08-29T15:00:00-05:00",
        },
    )

with sqlalchemy_engine.connect() as connection:
    run_count = connection.scalar(text("SELECT COUNT(*) FROM analysis_runs"))

assert run_count == 1

## 14. Metadata and SQLAlchemy Core preserve composability

Reflection reads the database's current schema. It is useful for inspection and legacy systems, but
it does not replace version-controlled schema declarations and migrations.

In [ ]:
database_inspector = inspect(sqlalchemy_engine)
reflected_table_names = set(database_inspector.get_table_names())

metadata = MetaData()
measurements_table = Table(
    "measurements",
    metadata,
    autoload_with=sqlalchemy_engine,
)
statement = (
    select(
        measurements_table.c.sensor_id,
        measurements_table.c.time_s,
        measurements_table.c.temperature_c,
    )
    .where(measurements_table.c.experiment_id == 1)
    .order_by(measurements_table.c.sensor_id, measurements_table.c.time_s)
)

with sqlalchemy_engine.connect() as connection:
    core_rows = connection.execute(statement).all()

assert "measurements" in reflected_table_names
assert len(core_rows) == 12

SQLAlchemy Core builds expression trees and binds values. The ORM maps rows and relationships to
Python objects and manages a unit of work. ORMs improve some application designs but can conceal SQL,
trigger N+1 query patterns, or load more data than intended. Data science often benefits from explicit
set-oriented SQL and DataFrames.

## 15. DuckDB is an in-process analytical database

DuckDB is designed for analytical, column-oriented workloads and can query DataFrames and local files
without running a separate server. It is excellent for local analytics and reproducible pipelines;
it is not a replacement for a multi-user transactional service in every application.

In [ ]:
duckdb_connection = duckdb.connect(":memory:")

all_measurements_frame = pd.read_sql_query(
    """
    SELECT
        m.measurement_id,
        e.material,
        m.time_s,
        m.sensor_id,
        s.distance_cm,
        m.temperature_c,
        m.quality_flag
    FROM measurements AS m
    JOIN experiments AS e USING (experiment_id)
    JOIN sensors AS s USING (sensor_id)
    ORDER BY m.measurement_id
    """,
    sqlite_connection,
)
duckdb_connection.register("measurement_frame", all_measurements_frame)

duckdb_summary = duckdb_connection.execute(
    """
    SELECT
        material,
        sensor_id,
        AVG(temperature_c) AS mean_temperature_c,
        COUNT(temperature_c) AS valid_measurements
    FROM measurement_frame
    GROUP BY material, sensor_id
    ORDER BY material, sensor_id
    """
).fetchdf()

assert duckdb_summary.shape == (6, 4)
assert duckdb_summary["valid_measurements"].sum() == 23
duckdb_summary

Registering the DataFrame exposes it read-only to this DuckDB connection. The query still needs an
explicit order if row order matters. Conversion back to pandas materializes results in Python memory.

### Analytical file formats can be queried without pandas

Parquet stores typed, columnar data and supports projection and predicate pushdown. CSV is text and
requires parsing/type inference. File layout, partitioning, statistics, compression, and schema
evolution affect performance and correctness.

In [ ]:
parquet_path = Path(database_workspace.name) / "measurements.parquet"
duckdb_connection.sql(
    "SELECT * FROM measurement_frame ORDER BY measurement_id"
).write_parquet(str(parquet_path))

parquet_relation = duckdb_connection.read_parquet(str(parquet_path))
parquet_count = parquet_relation.aggregate("COUNT(*) AS row_count").fetchone()[0]

assert parquet_path.is_file()
assert parquet_count == 24

Do not construct SQL file paths from untrusted text. Prefer library APIs and explicit path validation.
Treat downloaded database extensions and remote file access as supply-chain/network boundaries.

## 16. Scale begins with a working-set estimate

“Large” is not a fixed row count. It means the working set—input columns, decoded values,
intermediates, indexes, shuffle buffers, and output—does not fit safely within the memory, time,
network, or cost budget of the system doing the work. A 200 GB dataset may be easy if a partition
filter and three projected columns reduce it to 300 MB. A 4 GB table can still fail when a join,
sort, or one-hot encoding creates several much larger intermediates.

Estimate before loading. Measure a representative sample, include an expansion factor for temporary
objects, and leave headroom for Python, the notebook kernel, the operating system, and concurrent
work. An estimate is a decision aid, not a memory guarantee.

In [ ]:
sample_rows = len(all_measurements_frame)
sample_bytes = int(all_measurements_frame.memory_usage(index=True, deep=True).sum())
estimated_bytes_per_row = sample_bytes / sample_rows

hypothetical_rows = 100_000_000
intermediate_expansion_factor = 3.0
estimated_working_set_bytes = (
    estimated_bytes_per_row * hypothetical_rows * intermediate_expansion_factor
)
estimated_working_set_gib = estimated_working_set_bytes / 1024**3

print(f"Measured sample: {estimated_bytes_per_row:,.1f} bytes/row")
print(f"Rough working-set estimate: {estimated_working_set_gib:,.1f} GiB")

assert sample_rows == 24
assert estimated_bytes_per_row > 0
assert estimated_working_set_gib > 1

### Reduce data before changing machines

Use this order of questions:

1. **Can the source answer the question?** Push filters, joins, grouping, and projections into SQL,
   Athena, a warehouse, or a Parquet scanner.
2. **Can the algorithm be incremental?** Counts, sums, online means/variances, histograms, and many
   model updates can consume bounded batches.
3. **Can storage layout skip work?** Partitioning and Parquet row-group statistics may avoid files or
   blocks; columnar storage may avoid unused columns.
4. **Does one machine still suffice?** DuckDB, PyArrow, and Polars can process much more than a naïve
   pandas load, but local disk, memory, and CPU remain finite.
5. **Is distribution actually required?** Dask, Spark, Ray, a warehouse, or a managed query service
   adds scheduling, serialization, shuffle, debugging, and cost complexity. Use it for a measured
   reason.

`chunksize` bounds each input chunk; it does **not** make every downstream operation bounded. A global
sort, arbitrary join, exact median, or accumulating chunks into a list can still require the entire
working set.

### Recognize the accidental full-load workflow

```python
# Anti-pattern: transfer everything, then discard almost everything.
frame = pd.read_sql_query("SELECT * FROM measurements", connection)
small_answer = frame.loc[frame["quality_flag"] == "ok", ["sensor_id", "temperature_c"]]
```

The SQL service performs no reduction, the network carries unused rows and columns, pandas allocates
them, and the notebook becomes the least reliable component. Express the filter and projection in a
parameterized source query. Validate the returned schema and row count before expensive modeling.

## 17. A typical industry or laboratory data path

```text
instruments / applications / partner feeds
          │  immutable events, files, or transactional writes
          ▼
landing zone / operational database
          │  validation, cataloging, schema checks, quarantine
          ▼
object store data lake ───── metadata catalog
   raw → validated → curated        │
          │                         │
          ├── query engine / warehouse ── dashboards and governed SQL
          ├── batch or distributed compute ── reusable feature tables
          └── versioned extract ── training, evaluation, and model registry
```

In a lab, the sources might be microscopes, simulations, sensors, or an HPC cluster. In a company,
they might be product events, transactions, vendor feeds, or services. The engineering questions are
the same: ownership, one-record meaning, schema, units, identity, lateness, duplicates, access,
retention, lineage, validation, and replay.

### Raw, validated, and curated are contracts—not magic folder names

- **Landing/raw** preserves source evidence and ingestion metadata; restrict access and avoid silent
  mutation.
- **Validated** has parsed types, declared units, quarantined failures, and data-quality results.
- **Curated** has domain semantics, stable keys, documented joins, and consumer-facing contracts.
- **Feature/training data** adds a point-in-time rule, split policy, label definition, and immutable
  version or snapshot.

A pipeline should be restartable and idempotent. Write to a staging location, validate counts/schema,
publish an atomic pointer or catalog update when the platform supports it, and record a manifest.
“Exactly once” is an end-to-end property; a tool label alone does not establish it.

### AWS services solve different problems

| Service | Role in a common workflow | It is not |
| --- | --- | --- |
| Amazon S3 | durable object storage for files and data-lake objects | a mounted POSIX disk or relational database |
| AWS Glue Data Catalog | table/schema/partition metadata used by analytical services | the data itself |
| Amazon Athena | serverless SQL query engine over data such as files in S3 | an OLTP application database |
| Amazon RDS/Aurora | managed relational database service for transactional/client-server workloads | a reason to pull whole tables into pandas |
| Amazon Redshift | managed analytical warehouse | interchangeable with S3 object storage |
| EMR/Glue jobs/Batch | managed execution choices for larger transformations | automatically cheaper or simpler |

The non-AWS equivalents may be an institutional object store, PostgreSQL, a shared filesystem,
Trino, BigQuery, Snowflake, Databricks, Spark, Slurm, or Kubernetes. Learn the architectural role and
contract rather than memorizing one vendor's names.

## 18. Build a tiny partitioned Parquet lake locally

We now simulate a multi-file object-store dataset inside the temporary directory. The data remain
small enough for CI; the access patterns are the same ones used against much larger stores. Parquet
is typed and columnar. Hive-style directory partitions expose selected values in paths such as
`site=lab-a/material=copper/part-0.parquet`.

Partition columns should match common coarse filters. Partitioning by a near-unique identifier creates
tiny files and excessive metadata. Partitioning is not sorting, indexing, or a uniqueness constraint.

In [ ]:
measurement_lake_path = Path(database_workspace.name) / "measurement-lake"

scaled_frames = []
for acquisition_batch in range(48):
    batch = all_measurements_frame.copy()
    batch["event_id"] = range(
        acquisition_batch * len(batch),
        (acquisition_batch + 1) * len(batch),
    )
    batch["acquisition_batch"] = acquisition_batch
    batch["site"] = "lab-a" if acquisition_batch % 2 == 0 else "lab-b"
    batch["year"] = 2026
    scaled_frames.append(batch)

lake_frame = pd.concat(scaled_frames, ignore_index=True)
lake_table = pa.Table.from_pandas(lake_frame, preserve_index=False)
partition_schema = pa.schema([("site", pa.string()), ("material", pa.string())])

pads.write_dataset(
    lake_table,
    base_dir=measurement_lake_path,
    format="parquet",
    partitioning=pads.partitioning(partition_schema, flavor="hive"),
    basename_template="part-{i}.parquet",
    max_rows_per_file=500,
    max_rows_per_group=500,
)

parquet_files = sorted(measurement_lake_path.rglob("*.parquet"))
assert len(lake_frame) == 1_152
assert len(parquet_files) >= 4
assert all(path.is_file() for path in parquet_files)
print("Local lake files:", len(parquet_files))

### Projection, predicate pushdown, and partition pruning

- **Projection pushdown** asks storage to read only needed columns.
- **Predicate pushdown** moves a filter closer to the scan and may use Parquet statistics to skip row
  groups.
- **Partition pruning** skips whole files/directories when partition values cannot match.

Pushdown is an optimization opportunity, not a promise that zero irrelevant bytes are touched. File
metadata, filter support, statistics, encryption, filesystem behavior, and the execution engine all
matter. Inspect a plan and cloud scan metrics rather than inferring performance from source code.

In [ ]:
scientific_dataset = pads.dataset(
    measurement_lake_path,
    format="parquet",
    partitioning="hive",
)

lake_filter = (pads.field("site") == "lab-a") & (
    pads.field("quality_flag") == "ok"
)
lake_scanner = scientific_dataset.scanner(
    columns=["event_id", "sensor_id", "temperature_c", "site", "material"],
    filter=lake_filter,
    batch_size=128,
    use_threads=False,
)

batch_sizes = []
temperature_sum = 0.0
temperature_count = 0
for record_batch in lake_scanner.to_batches():
    temperatures = record_batch.column("temperature_c")
    batch_sizes.append(record_batch.num_rows)
    temperature_sum += pc.sum(temperatures).as_py() or 0.0
    temperature_count += pc.count(temperatures).as_py()

arrow_mean_temperature = temperature_sum / temperature_count
expected_mean_temperature = lake_frame.loc[
    (lake_frame["site"] == "lab-a")
    & (lake_frame["quality_flag"] == "ok"),
    "temperature_c",
].mean()

assert batch_sizes
assert max(batch_sizes) <= 128
assert temperature_count < len(lake_frame)
assert isclose(arrow_mean_temperature, expected_mean_temperature, rel_tol=1e-12)
print("Bounded batches:", batch_sizes)
print("Incremental mean:", round(arrow_mean_temperature, 3))

The loop consumes each Arrow `RecordBatch` and keeps only a sum and count. It never concatenates the
batches. This is a bounded-state algorithm. In production, also decide what happens after a partial
failure: which object/version and batch were completed, whether output is idempotent, and how the run
will resume without double-counting.

Calling `to_table()` or `list(scanner.to_batches())` would materialize the selected result. That may be
correct after aggressive reduction; make it an explicit boundary.

### Polars builds a lazy query plan

`scan_parquet` returns a `LazyFrame`; it describes work rather than immediately loading all files.
Filters and selected columns can be optimized into the scan. `collect` is the materialization/action
boundary. Streaming-capable execution can reduce peak memory, but not every operation is streamable.
Always inspect the optimized plan and benchmark a representative workload.

In [ ]:
lake_glob = str(measurement_lake_path / "**" / "*.parquet")
lazy_summary = (
    pl.scan_parquet(lake_glob, hive_partitioning=True)
    .filter((pl.col("site") == "lab-a") & (pl.col("quality_flag") == "ok"))
    .group_by(["material", "sensor_id"])
    .agg(
        pl.col("temperature_c").mean().alias("mean_temperature_c"),
        pl.col("temperature_c").count().alias("valid_measurements"),
    )
    .sort(["material", "sensor_id"])
)

optimized_plan = lazy_summary.explain(optimized=True)
polars_summary = lazy_summary.collect(engine="streaming")

print(optimized_plan)
assert isinstance(lazy_summary, pl.LazyFrame)
assert "Parquet SCAN" in optimized_plan
assert polars_summary.height == 6
assert polars_summary["valid_measurements"].sum() == temperature_count
polars_summary

### DuckDB can query the same files directly

DuckDB is often an excellent bridge between laptop-scale exploration and warehouse SQL. It can scan
Parquet, prune columns and filters, aggregate locally, and return only the small answer to pandas.
The result below is materialized; the full lake is not first copied into a DataFrame.

In [ ]:
duckdb_lake_relation = duckdb_connection.read_parquet(
    lake_glob,
    hive_partitioning=True,
)
duckdb_lake_summary = (
    duckdb_lake_relation.filter("site = 'lab-a' AND quality_flag = 'ok'")
    .aggregate(
        "material, sensor_id, AVG(temperature_c) AS mean_temperature_c, "
        "COUNT(temperature_c) AS valid_measurements",
        "material, sensor_id",
    )
    .order("material, sensor_id")
    .fetchdf()
)

assert duckdb_lake_summary.shape == (6, 4)
assert duckdb_lake_summary["valid_measurements"].sum() == temperature_count
assert isclose(
    duckdb_lake_summary["mean_temperature_c"].mean(),
    polars_summary["mean_temperature_c"].mean(),
    rel_tol=1e-12,
)
duckdb_lake_summary

### File layout is part of performance engineering

Avoid the two extremes: one enormous indivisible file and millions of tiny files. File and row-group
sizes affect parallelism, metadata requests, retries, pruning, and memory. Compact small outputs,
choose partitions from real query patterns, and record schema/partition changes. Never assume an
object-store listing is a transactional snapshot; use a manifest, catalog/table format, or platform
snapshot mechanism when consistency matters.

## 19. Bounded database access: pages and server-side cursors

`fetchall()` and ordinary pandas reads materialize the result. A DB-API cursor can fetch bounded
batches, and some remote drivers offer named/server-side cursors that avoid buffering the entire
result on the client. The transaction remains open while streaming, so long processing can retain a
snapshot and server resources. Close promptly and understand the driver's buffering behavior.

For resumable ordered extraction, keyset pagination (“rows after the last stable key”) is usually more
stable than increasing `OFFSET`, which repeatedly skips work and behaves poorly under concurrent
changes. The ordering key must be unique and the snapshot/mutation policy explicit.

In [ ]:
def iter_measurement_pages(
    connection: sqlite3.Connection,
    *,
    page_size: int,
) -> Iterator[list[sqlite3.Row]]:
    """Yield measurements in bounded, stable-key pages.

    Parameters
    ----------
    connection
        Open SQLite connection using ``sqlite3.Row`` values.
    page_size
        Positive maximum number of rows returned per page.

    Yields
    ------
    list of sqlite3.Row
        One page ordered by the unique measurement identifier.

    Raises
    ------
    ValueError
        If ``page_size`` is not positive.
    """
    if page_size <= 0:
        raise ValueError("page_size must be positive")

    last_measurement_id = 0
    while True:
        page = connection.execute(
            """
            SELECT measurement_id, sensor_id, temperature_c
            FROM measurements
            WHERE measurement_id > ?
            ORDER BY measurement_id
            LIMIT ?
            """,
            (last_measurement_id, page_size),
        ).fetchall()
        if not page:
            return
        yield page
        last_measurement_id = page[-1]["measurement_id"]


measurement_pages = list(iter_measurement_pages(sqlite_connection, page_size=5))
page_ids = [row["measurement_id"] for page in measurement_pages for row in page]

assert [len(page) for page in measurement_pages] == [5, 5, 5, 5, 4]
assert page_ids == list(range(1, 25))

For Psycopg, a production pattern may use a named server-side cursor and `fetchmany`, but it must run
inside a real connection/transaction and be tested against PostgreSQL:

```python
with psycopg.connect(approved_dsn) as connection:
    with connection.cursor(name="measurement_stream") as cursor:
        cursor.itersize = 10_000
        cursor.execute(parameterized_query, query_parameters)
        while rows := cursor.fetchmany(10_000):
            process_bounded_batch(rows)
```

Do not run slow model inference while holding this transaction open. A common production design first
publishes a versioned extract, closes the database transaction, and then trains from that immutable
artifact.

## 20. AWS object-storage workflows with Boto3

S3 stores objects addressed by bucket and key; it does not provide ordinary filesystem behaviors such
as atomic rename, directories, or in-place append. A “folder” is a key-prefix convention. Production
code must handle pagination, retries, version identity, checksums, access policy, encryption, and
partial transfers.

Prefer workload identity: an IAM role attached to approved compute, AWS IAM Identity Center/SSO for a
person, or another short-lived provider in the standard credential chain. Do not paste access keys
into a notebook, source file, connection URL, or shared environment file. Region, account, role,
bucket, and prefix should be explicit configuration with least-privilege permissions.

In [ ]:
def split_s3_uri(uri: str) -> tuple[str, str]:
    """Split an S3 URI into a bucket and nonempty object key.

    Parameters
    ----------
    uri
        URI of the form ``s3://bucket/key``.

    Returns
    -------
    tuple of str
        Bucket and object key.

    Raises
    ------
    ValueError
        If the URI is not an unambiguous S3 object URI.
    """
    parsed = urlsplit(uri)
    key = parsed.path.lstrip("/")
    if parsed.scheme != "s3" or not parsed.netloc or not key:
        raise ValueError("expected s3://bucket/nonempty-key")
    if parsed.query or parsed.fragment:
        raise ValueError("query strings and fragments are not object keys")
    return parsed.netloc, key


example_bucket, example_key = split_s3_uri(
    "s3://rice-dsm-course-example/validated/measurements/part-000.parquet"
)
assert example_bucket == "rice-dsm-course-example"
assert example_key.endswith("part-000.parquet")

### Test cloud code without a cloud account

The AWS SDK separates client construction from API calls. This client is unsigned, and `Stubber`
intercepts the operation locally. It verifies the expected request shape and returns a modeled
response; no DNS, credential lookup, account, charge, or network connection is involved.

Unit tests should stub narrow client behavior. Separate integration tests should run against an
approved test account/bucket with lifecycle cleanup and cost limits. A stub proves application logic,
not IAM policies, service quotas, network routes, encryption configuration, or AWS semantics end to
end.

In [ ]:
s3_client = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED),
)

expected_list_parameters = {
    "Bucket": "rice-dsm-course-example",
    "Prefix": "validated/measurements/",
    "MaxKeys": 1000,
}
stubbed_list_response = {
    "IsTruncated": False,
    "Name": "rice-dsm-course-example",
    "Prefix": "validated/measurements/",
    "MaxKeys": 1000,
    "KeyCount": 2,
    "Contents": [
        {"Key": "validated/measurements/part-000.parquet", "Size": 2048},
        {"Key": "validated/measurements/part-001.parquet", "Size": 3072},
    ],
}

with Stubber(s3_client) as s3_stubber:
    s3_stubber.add_response(
        "list_objects_v2",
        stubbed_list_response,
        expected_list_parameters,
    )
    listed_objects = s3_client.list_objects_v2(**expected_list_parameters)

listed_keys = [item["Key"] for item in listed_objects.get("Contents", [])]
assert listed_objects["IsTruncated"] is False
assert len(listed_keys) == 2
assert sum(item["Size"] for item in listed_objects["Contents"]) == 5120

Real listings can be truncated: use the Boto3 paginator rather than assuming one
`list_objects_v2` response is complete. For large transfers, `download_file`/`download_fileobj` uses
the managed transfer layer; `get_object` returns a streaming body and supports byte ranges. Neither
choice means an analytical program should download every object. Prefer a dataset/query engine that
can read selected Parquet columns and partitions in place.

Record the S3 URI **and** immutable version identifier when versioning is enabled, or a published
manifest/checksum. An ETag is not universally the MD5 of object content. Close response bodies and
clients, use timeouts/retry limits, and avoid printing signed URLs or sensitive object keys.

### Athena pushes SQL to data in S3

Athena is useful when an analyst needs a small relational answer from a large cataloged dataset.
Submission is asynchronous: start a query, retain its query-execution ID, poll with bounded backoff,
handle terminal success/failure/cancellation states, and consume paginated results or the output
artifact. Configure a workgroup, database/catalog, encrypted output location, scan limits, and tags.

The following pure function builds a request but never creates or calls an Athena client. SQL text
must still be reviewed and parameterized through an appropriate API/workflow; IAM is not protection
against an accidentally expensive full scan.

In [ ]:
def build_athena_request(
    *,
    query: str,
    database: str,
    output_uri: str,
    idempotency_token: str,
) -> dict[str, object]:
    """Build a reviewable Athena query-submission request without sending it."""
    if not query.strip().lower().startswith("select"):
        raise ValueError("this teaching workflow permits read-only SELECT queries")
    split_s3_uri(output_uri)
    if len(idempotency_token) < 32:
        raise ValueError("idempotency_token must contain at least 32 characters")
    return {
        "QueryString": query,
        "QueryExecutionContext": {"Database": database},
        "ResultConfiguration": {"OutputLocation": output_uri},
        "WorkGroup": "course-analytics",
        "ClientRequestToken": idempotency_token,
    }


athena_request = build_athena_request(
    query="""
        SELECT material, AVG(temperature_c) AS mean_temperature_c
        FROM validated_measurements
        WHERE site = 'lab-a' AND year = 2026
        GROUP BY material
    """,
    database="course_science",
    output_uri="s3://rice-dsm-course-example/query-results/",
    idempotency_token="lecture-06-example-token-000000001",
)

assert athena_request["WorkGroup"] == "course-analytics"
assert "SELECT *" not in str(athena_request["QueryString"]).upper()
assert "site = 'lab-a'" in str(athena_request["QueryString"])

### Cost, security, and observability are query requirements

For an object-store query, record bytes scanned, files/partitions touched, execution time, result
rows, query ID, workgroup/warehouse, and estimated cost. Compression, column projection, partition
pruning, and compacted file layout can reduce scan cost. `LIMIT 10` often limits returned rows without
guaranteeing only a tiny amount of source data is scanned.

Protect sensitive data with least-privilege IAM, bucket policies, encryption, private networking where
required, audit logs, retention/deletion rules, and separate raw/curated access. Never copy governed
data to a laptop merely because Python can authenticate to it.

## Choosing the smallest tool that safely fits

| Need | Good first candidate | Important boundary |
| --- | --- | --- |
| Small, already-reduced tabular result | pandas | eagerly materializes most operations |
| Typed multi-file scans and record batches | PyArrow Dataset | low-level; you own algorithm state |
| Lazy single-machine DataFrame transformations | Polars | inspect plans; not every operation streams |
| Local SQL over Parquet/CSV/DataFrames | DuckDB | one machine and local resource limits |
| Transactional shared state | PostgreSQL/RDS-style DBMS | connection, concurrency, and schema operations |
| SQL over a data lake | Athena/Trino-style engine | scan cost, catalog, file layout, async jobs |
| Parallel/distributed Python DataFrames | Dask | partitions, scheduler, shuffle, cluster memory |
| Large distributed SQL/ETL | Spark/warehouse | operational complexity and data movement |

Library choice comes after the access pattern. Switching pandas syntax to a distributed DataFrame
does not repair an unnecessary `SELECT *`, a many-to-many join explosion, leakage, bad partitions,
or an irreproducible mutable source.

## Guided practice: design a 2 TB training-data workflow

A laboratory has 2 TB of partitioned Parquet measurements in S3, 30 GB arriving daily, and a model
that needs six columns from one site over the last 90 days.

1. Draw the landing, validation, catalog, query, extract, training, and registry boundaries.
2. State which columns and partitions should be read and where filtering/aggregation occurs.
3. Decide whether pandas, PyArrow, Polars, DuckDB, Athena, Dask, or Spark belongs at each boundary.
4. Define snapshot identity, schema/unit checks, duplicate policy, and point-in-time split.
5. Define IAM roles, encryption, audit, retention, retry/idempotency, and cost metrics.
6. Explain how a failed run resumes without duplicating the published extract.

**Success criterion:** the laptop never downloads 2 TB, and another engineer can reproduce the exact
training rows from the manifest.

## Independent practice: bounded aggregation

Adapt the Arrow scanner to compute count, mean, and sample variance using bounded state. Then compare
the result with pandas on this small fixture only.

- Do not call `to_table`, `to_pandas`, `list(...)`, or concatenate batches.
- Specify null behavior and numerical stability.
- Track maximum observed batch size.
- Add tests for an empty selection, all-null values, and two different batch sizes.

**Success criterion:** peak algorithm state does not grow with row count, and independent results
agree within a justified tolerance.

## Extension: distributed execution is a systems exercise

Run the same group-by with Dask or Spark only after profiling the single-machine plan. Predict the
partition graph, serialization, network shuffle, skew, task count, and failure-recovery behavior.
Compare wall time and total compute—not only whether the distributed version runs.

In an HPC lab, workers may use Slurm and a shared parallel filesystem instead of cloud services.
Avoid thousands of workers appending to one CSV or SQLite file. Prefer independent immutable outputs,
then validate and compact/reduce them through an explicit job. Record scheduler job ID, code commit,
environment, input manifest, resource request, random seeds, and output checksums.

## 21. Remote databases add a network and identity boundary

For a PostgreSQL-style system:

```text
application
  → SQLAlchemy Engine / pool
  → Psycopg driver
  → TLS network connection
  → PostgreSQL server
  → role, database, schema, tables, indexes, storage
```

New failure modes include DNS, routing, certificate validation, authentication, authorization, server
capacity, pool exhaustion, timeouts, failover, and connections interrupted after an unknown commit
state.

### Build URLs programmatically and keep secrets out of source

`URL.create` handles URL components without manual escaping. This offline example intentionally omits
a password and uses the reserved `.invalid` domain. In production, obtain credentials from an approved
secret manager, workload identity, or short-lived token; never commit them or print the full URL.

In [ ]:
remote_database_url = URL.create(
    "postgresql+psycopg",
    username="course_reader",
    password=None,
    host="database.example.invalid",
    port=5432,
    database="scientific_measurements",
)

safe_remote_description = remote_database_url.render_as_string(
    hide_password=True
)
print("Offline configuration example:", safe_remote_description)

assert remote_database_url.drivername == "postgresql+psycopg"
assert remote_database_url.host == "database.example.invalid"
assert remote_database_url.password is None
assert "password" not in safe_remote_description.lower()

Never place a real credential in a notebook cell, connection string committed to Git, traceback,
shell history, chart, or log. Environment variables are better than source code but may still be
visible to processes and debugging tools. Follow the institution's secret-management policy.

### Engine construction is lazy; connection is the network event

Creating the following Engine loads the PostgreSQL dialect and Psycopg driver but does not call
`.connect()`. The `.invalid` host guarantees that accidental real infrastructure is not targeted.

In [ ]:
remote_engine = create_engine(
    remote_database_url,
    pool_size=5,
    max_overflow=2,
    pool_timeout=10,
    pool_pre_ping=True,
    connect_args={
        "connect_timeout": 5,
        "sslmode": "require",
        "application_name": "rice-dsm-course",
    },
)

assert remote_engine.dialect.name == "postgresql"
assert remote_engine.dialect.driver == "psycopg"
assert remote_engine.pool.size() == 5
remote_engine.dispose()

`sslmode="require"` encrypts but may not provide the strongest identity verification policy for every
deployment. Use the server/provider's certificate guidance. Connection, statement, lock, and pool
timeouts control different waits; define each according to the operation and service objective.

## 22. Connection pools are bounded shared resources

A pool reuses expensive connections and limits concurrency against the server. It is not a cache of
query results. Size it from application concurrency and database capacity, not “as large as possible.”

Always return connections through context managers. Long transactions retain connections and may
hold locks or old row versions. After process forks, dispose/recreate pools according to the library's
guidance. Serverless platforms may need an external pooler or different configuration.

### Do not retry every failure

Retry only failures classified as transient and only when repeating the operation is safe. A lost
connection after sending `COMMIT` can leave the client uncertain whether the transaction committed.
Blindly retrying an insert may duplicate effects.

Use idempotency keys, unique constraints, bounded attempts, exponential backoff with jitter, deadlines,
and observable final failure. Syntax errors, permission denials, constraint violations, and invalid
credentials generally need correction rather than retry.

## 23. Transaction isolation governs concurrent observations

Atomicity does not mean every concurrent transaction sees the same world. Isolation policies govern
phenomena such as nonrepeatable reads and phantoms. Stronger isolation can reduce anomalies but add
blocking, aborts, or retry requirements.

PostgreSQL's `READ COMMITTED`, `REPEATABLE READ`, and `SERIALIZABLE` have documented engine-specific
semantics. Choose based on invariants and access patterns; do not infer behavior from the isolation
level name alone.

For a reproducible analytical extract, a repeatable snapshot may matter more than reading the newest
row. For an operational dashboard, freshness may dominate. Record snapshot/watermark semantics and
avoid holding long-running transactions that harm the production system.

## 24. Migrations version the schema

A migration is an ordered, reviewable transition such as adding a nullable column, backfilling it,
validating it, and later enforcing non-nullability. Production-safe change often uses **expand and
contract**:

1. add backward-compatible schema;
2. deploy code that understands old and new forms;
3. backfill and validate in bounded batches;
4. switch readers/writers;
5. remove the old form only after dependents migrate.

Back up before risky change, rehearse on representative size, monitor locks and runtime, and test
rollback or forward-recovery. Tools such as Alembic coordinate SQLAlchemy migrations; `create_all()`
is not a migration history.

## 25. Reproducible extracts require more than SQL text

Record at least:

- database/service and logical dataset identity;
- query text or versioned query name;
- bound parameter values excluding secrets;
- schema/migration version;
- transaction snapshot, data version, or ingestion watermark;
- time zone and timestamp interpretation;
- explicit column and row ordering when relevant;
- row count, key uniqueness, missingness, and range checks;
- extraction time and responsible code version; and
- a checksum or immutable artifact identity when policy permits.

`SELECT ... FROM measurements` against a mutable table is not a reproducible dataset specification.

### Machine-learning boundaries belong in the query contract

Split by entity and time before fitting transformations. A query such as “all rows with timestamp
before cutoff” may still leak when the same patient/device/entity spans both sides, features were
backfilled from the future, or labels use a later outcome.

In [ ]:
split_query = """
SELECT
    measurement_id,
    experiment_id,
    sensor_id,
    time_s,
    temperature_c,
    CASE
        WHEN time_s < :evaluation_start_s THEN 'train'
        ELSE 'evaluation'
    END AS split_name
FROM measurements
ORDER BY experiment_id, sensor_id, time_s
"""

split_frame = pd.read_sql_query(
    split_query,
    sqlite_connection,
    params={"evaluation_start_s": 60},
)

assert split_frame.loc[
    split_frame["split_name"] == "train", "time_s"
].max() < 60
assert split_frame.loc[
    split_frame["split_name"] == "evaluation", "time_s"
].min() >= 60

This time split intentionally allows the same experiment and sensor in both partitions. That may be
correct for forecasting later times for known experiments, but wrong for generalizing to unseen
experiments. Split policy follows the deployment question, not a universal recipe.

## 26. Security, privacy, and governance

- use least-privilege roles: readers should not own or drop production tables;
- separate application, migration, analyst, and administrative identities;
- restrict network paths and require approved TLS verification;
- rotate/revoke credentials and prefer short-lived identity;
- parameterize values and allowlist identifiers;
- classify sensitive columns and minimize extraction;
- avoid sensitive values in logs, exceptions, notebook outputs, caches, and test fixtures;
- audit access and destructive changes;
- define retention, deletion, backup, and restoration policy; and
- treat database dumps and analytical extracts as sensitive copies.

Row-level security and masking help only when correctly configured and tested.

## 27. Observability and cost

Measure connection acquisition time, pool occupancy, query latency, rows scanned/returned, transaction
duration, lock waits, errors by class, retries, and resource use. Attach safe query identifiers rather
than raw SQL containing values. Distributed traces can connect an application request to database work.

For warehouses billed by bytes or compute, an accidental unbounded scan is both performance and cost
risk. Use query limits, partitions, clustering/indexes, budgets, and review of execution plans.

## 28. Testing database code at the correct boundary

- **unit tests:** pure query builders, validators, and row-to-domain conversions;
- **SQLite/DuckDB integration tests:** disposable local schema and transaction behavior when those
  engines are supported targets;
- **production-engine integration tests:** real PostgreSQL/MySQL container or test service for dialect,
  constraints, isolation, extensions, and migrations;
- **contract tests:** expected columns, types, keys, permissions, and producer/consumer behavior;
- **migration tests:** upgrade from supported prior schemas and validate data preservation;
- **performance tests:** representative volume and query plan in a controlled environment; and
- **failure tests:** timeouts, deadlocks/serialization failures, unavailable service, and uncertain
  commit outcomes where the architecture can simulate them.

A SQLite test cannot prove PostgreSQL behavior. A mocked cursor cannot prove that SQL parses.

In [ ]:
def count_measurements_for_material(
    connection: sqlite3.Connection,
    material: str,
) -> int:
    """Count stored measurements associated with one material.

    Parameters
    ----------
    connection : sqlite3.Connection
        Open connection whose schema contains experiments and measurements.
    material : str
        Material name bound as a query value.

    Returns
    -------
    int
        Number of joined measurement rows.
    """

    row = connection.execute(
        """
        SELECT COUNT(*)
        FROM measurements AS m
        JOIN experiments AS e USING (experiment_id)
        WHERE e.material = ?
        """,
        (material,),
    ).fetchone()
    if row is None:
        raise RuntimeError("count query returned no row")
    return int(row[0])


assert count_measurements_for_material(sqlite_connection, "copper") == 12
assert count_measurements_for_material(sqlite_connection, "unknown") == 0

The function test checks actual SQL, binding, joins, and the local schema. If the production target is
PostgreSQL, repeat important integration cases there. Test doubles remain useful for rare network
errors but should not become the only database evidence.

## 29. Debugging a database failure

1. classify the phase: URL/configuration, DNS/network, authentication, authorization, connection pool,
   transaction, SQL syntax, constraint, timeout, result conversion, or application validation;
2. preserve the exception type, cause chain, safe database host/service identifier, and query name;
3. never print secrets or sensitive bound values;
4. reduce to the smallest parameterized query and known schema version;
5. verify the active database, schema/search path, role, and transaction state;
6. inspect row counts, key uniqueness, nulls, types, and ordering;
7. compare dialect/version behavior and obtain a representative query plan;
8. determine whether retry is safe and bounded;
9. close/return failed connections as the driver requires; and
10. add a regression or integration test at the boundary that actually failed.

### Common failure modes

| Symptom | Likely cause | First check |
| --- | --- | --- |
| works locally, fails remotely | SQL dialect/schema/permission difference | engine, version, role, search path |
| query returns too many rows | many-to-many join or duplicate key | key counts before join |
| query returns inconsistent order | missing `ORDER BY` | output contract |
| missing rows vanish | inner join or `WHERE` predicate on nullable side | join type and null logic |
| `= NULL` finds nothing | SQL three-valued logic | use `IS NULL` |
| partial multi-table update | missing transaction boundary | commit/rollback path |
| pool timeout | leaked or long-held connections | context managers and transaction duration |
| repeated insert after retry | uncertain commit and non-idempotent write | unique/idempotency key |
| pandas exhausts memory | unbounded materialization | projection, filtering, chunks, aggregation |
| model quality changes | mutable extract or schema drift | query/version/snapshot/features |
| index unused | low selectivity, column order, casts, stale statistics | representative plan |
| secret appears in logs | URL/parameters printed | redaction and logging boundary |

## Guided practice: repair an unsafe query

Review this code without running it:

```python
column = input("Column: ")
minimum = input("Minimum: ")
sql = f"SELECT * FROM measurements WHERE {column} >= {minimum}"
```

1. Identify separate risks for the identifier and value.
2. Allow only `time_s` or `temperature_c` through an application-owned mapping.
3. Bind the minimum as a parameter.
4. Select explicit output columns and order the result.
5. Test hostile identifier/value strings.

**Success criterion:** user text cannot change SQL structure, and the output schema/order is explicit.

In [ ]:
allowed_filter_columns = {
    "time": "time_s",
    "temperature": "temperature_c",
}
requested_filter = "temperature"
minimum_value = 50.0

# The application owns every SQL fragment; the external value chooses only a key.
filter_column_sql = allowed_filter_columns[requested_filter]
practice_sql = f"""
SELECT measurement_id, sensor_id, time_s, temperature_c
FROM measurements
WHERE {filter_column_sql} >= ?
ORDER BY measurement_id
"""
practice_rows = sqlite_connection.execute(
    practice_sql,
    (minimum_value,),
).fetchall()

assert practice_rows
assert all(row["temperature_c"] >= minimum_value for row in practice_rows)

## Guided practice: predict join and aggregate results

Add a fourth sensor with no measurements and answer before executing:

1. How many rows result from an inner join of sensors to measurements?
2. How many from a left join beginning with sensors?
3. What do `COUNT(*)` and `COUNT(temperature_c)` return for the unused sensor group?
4. Where should the null-producing side of the join appear?

**Success criterion:** draw the keys, predict exact row counts, then verify with parameterized SQL.

## Independent practice: design a model registry schema

Design normalized tables for models, immutable model versions, training datasets, evaluation runs,
metrics, and deployment decisions. Specify:

- natural versus surrogate keys;
- foreign keys and deletion behavior;
- uniqueness and nullability;
- artifact URI plus checksum;
- code, environment, query, and dataset snapshot identity;
- metric name, value, unit/direction, slice, and evaluation population;
- approval actor/time and decision status; and
- fields that require access control or retention policy.

**Success criterion:** one model can have many immutable versions and evaluations without overwriting
history, and every deployed artifact traces to training and evaluation evidence.

## Independent practice: a reproducible analytical extract

Write a function that accepts an SQLAlchemy `Connection`, material, start/end time, and sensor IDs.
Return a DataFrame plus an immutable manifest containing query version, non-secret parameters, ordered
columns, row count, key uniqueness, schema version, and extraction timestamp.

Test normal, empty, invalid interval, unknown sensor, duplicate-key, null-temperature, and rollback
cases. Do not interpolate values or identifiers.

**Success criterion:** another process can determine exactly what was requested and validate what was
returned without receiving credentials.

## Extension: local-to-remote migration review

Suppose a prototype moves from SQLite to managed PostgreSQL. Produce a migration plan covering:

- type and SQL-dialect differences;
- foreign-key and constraint enforcement;
- timestamp/time-zone semantics;
- identity/sequence generation;
- transaction and concurrency assumptions;
- roles, grants, TLS, secret distribution, and rotation;
- connection pool and timeout policy;
- schema migrations and backward compatibility;
- production-engine integration tests;
- backup, restore, monitoring, and incident ownership; and
- cutover validation and rollback.

**Success criterion:** the plan does not assume passing SQLite tests proves PostgreSQL behavior.

## Extension: query-review checklist

For one real analytical query, record:

1. scientific question and one-row meaning;
2. source tables, owners, and freshness;
3. selected columns and units;
4. join keys/cardinality and unmatched-row policy;
5. filters, time zone, null policy, and denominators;
6. aggregation level and window definitions;
7. leakage and sensitive-data risks;
8. expected row count/range and uniqueness;
9. representative query plan and cost; and
10. snapshot/reproducibility evidence.

**Success criterion:** a reviewer can challenge both scientific and operational assumptions without
reverse-engineering the SQL.

## 30. Release disposable resources

Connections are resources, not ordinary values. Close them deterministically. SQLAlchemy `dispose()`
closes pooled connections; the SQLite connection context manager used earlier did not close it.

In [ ]:
s3_client.close()
duckdb_connection.close()
sqlalchemy_engine.dispose()
sqlite_connection.close()

database_workspace.cleanup()

assert not database_path.exists()
assert not parquet_path.exists()
assert not measurement_lake_path.exists()

## Retrieval practice

Answer without executing code:

1. Distinguish database, DBMS, driver, connection, cursor/result, transaction, and Engine.
2. When is SQLite a good fit, and which remote-database behaviors can it not establish?
3. Why do value placeholders not solve dynamic table/column identifier safety?
4. What do primary, foreign, unique, not-null, and check constraints protect?
5. Why does `column = NULL` not find missing rows?
6. Compare `COUNT(*)`, `COUNT(column)`, and `AVG(column)` with nulls.
7. Why can a join multiply rows, and which checks expose it?
8. What does a transaction context do when an exception escapes?
9. Why can blindly retrying a write duplicate effects?
10. What does a connection pool control?
11. Which evidence makes an analytical extract reproducible?
12. Distinguish projection pushdown, predicate pushdown, and partition pruning.
13. Why does input chunking not make every downstream algorithm memory-bounded?
14. Distinguish S3, Glue Catalog, Athena, RDS, and Redshift by architectural role.
15. Where is the materialization boundary in PyArrow, Polars, DuckDB, and pandas?
16. Why must important tests run against the production database and cloud services?

## Takeaway

```text
scientific question and working-set estimate
  → reduce at the source: projection, filters, partitions, aggregation
  → normalized identities and constraints
  → parameterized, transaction-aware query
  → explicit joins, nulls, aggregation, and order
  → validated result and reproducibility manifest
  → least-privilege operation, monitoring, and tested change
```

PyArrow record batches, Polars lazy plans, and DuckDB file scans keep reduction close to storage; AWS-style object storage and query services add identity, cost, catalog, and distributed-systems boundaries. SQLite teaches durable local transactions with almost no setup. DuckDB brings analytical SQL to local
files and DataFrames. SQLAlchemy organizes engines, dialects, connections, and transactions. Psycopg
crosses the PostgreSQL client/server boundary. None removes the need to understand schema, SQL,
concurrency, security, or the scientific meaning of the extracted rows.

The next notebook returns to deeper NumPy mechanics that support efficient numerical algorithms.

## Further reading

- [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html)
- [SQLite foreign keys](https://www.sqlite.org/foreignkeys.html)
- [SQLAlchemy unified tutorial](https://docs.sqlalchemy.org/en/20/tutorial/)
- [SQLAlchemy engines and URLs](https://docs.sqlalchemy.org/en/20/core/engines.html)
- [SQLAlchemy transactions](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html)
- [SQLAlchemy connection pooling](https://docs.sqlalchemy.org/en/20/core/pooling.html)
- [DuckDB Python API](https://duckdb.org/docs/stable/clients/python/overview)
- [DuckDB SQL on pandas](https://duckdb.org/docs/stable/guides/python/sql_on_pandas.html)
- [Psycopg basic usage](https://www.psycopg.org/psycopg3/docs/basic/usage.html)
- [PostgreSQL constraints](https://www.postgresql.org/docs/current/ddl-constraints.html)
- [PostgreSQL transaction isolation](https://www.postgresql.org/docs/current/transaction-iso.html)
- [Apache Arrow Dataset API](https://arrow.apache.org/docs/python/dataset.html)
- [PyArrow Scanner](https://arrow.apache.org/docs/python/generated/pyarrow.dataset.Scanner.html)
- [Polars lazy API](https://docs.pola.rs/user-guide/lazy/)
- [Polars streaming](https://docs.pola.rs/user-guide/concepts/streaming/)
- [Boto3 S3](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)
- [Boto3 credentials](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/credentials.html)
- [Amazon Athena](https://docs.aws.amazon.com/athena/latest/ug/what-is.html)
- [Dask DataFrame best practices](https://docs.dask.org/en/stable/dataframe-best-practices.html)
